# Next-Observed Indicator Forecasting (v3)

Forecasts the probability that each threat indicator (IP address) will be
**re-observed** at a given operating division (OpDiv) within the next
**1, 7, 14, 30, and 45 days**, based on its recent observation history.

**Pipeline**

1. **Load** — concatenate the daily per-OpDiv observation extracts for the last 100 days.
2. **Dense panel** — expand each OpDiv's history into a complete `source x date x indicator`
   grid so that days with *no* observation become explicit zeros.
3. **Features** — per indicator: recency, windowed frequencies, inter-arrival gap
   statistics, and burstiness.
4. **Models** — four complementary probability estimates per horizon:
   | Model | Captures |
   |---|---|
   | Logistic regression | Linear baseline probabilities |
   | Gradient-boosted trees | Non-linear patterns and feature interactions |
   | Exponential (Poisson) | Memoryless rate-based re-occurrence |
   | Weibull AFT (survival) | Time-to-event behavior and burstiness |
5. **Ensemble** — fixed-weight blend of the four estimates, mapped to analyst-facing
   confidence tiers.

> **Note:** internal file paths have been redacted (`<PATH_REMOVED_FOR_PRIVACY>`), so this
> notebook is a read-only snapshot — cell outputs are preserved from the original run.


## 1 · Setup

In [ ]:
import os
import pickle
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import WeibullAFTFitter
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Known-noisy warnings for this pandas/sklearn combination; safe to silence here.
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message="DataFrameGroupBy.apply operated on the grouping columns",
)
warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)


In [ ]:
# ----------------------------------------------------------------------------- config
LOOKBACK_DAYS = 100                      # history window pulled into the model
HORIZONS = [1, 7, 14, 30, 45]            # forecast horizons (days)
LABEL_HORIZONS = [7, 14, 30, 45]         # horizons with a direct label in the snapshot

# Ensemble blend weights (sum to 1.0)
ENSEMBLE_WEIGHTS = {"logistic": 0.30, "gbt": 0.25, "weibull": 0.25, "exponential": 0.20}

DATA_PATH_TEMPLATE = "<PATH_REMOVED_FOR_PRIVACY>/htoc_opdiv_obs_d{date}.csv"

today = datetime.today()
end_dt = today
start_dt = today - timedelta(days=LOOKBACK_DAYS)

print(today)
print(end_dt.strftime("%Y-%m-%d"))


2025-10-17 11:47:46.739364
2025-10-17


## 2 · Load daily observation extracts

One CSV per day, one row per `(indicator, source, OpDiv)` with an observation count.
Missing days are skipped with a warning rather than failing the whole run.


In [ ]:
def load_observation_files(paths):
    """Read every existing CSV in `paths`, skipping (and reporting) missing days."""
    frames = []
    for path in paths:
        if not os.path.exists(path):
            print(f"File {path} does not exist. Skipping.")
            continue
        frames.append(pd.read_csv(path))
    return frames


daily_paths = [
    DATA_PATH_TEMPLATE.format(date=dt.strftime("%Y%m%d"))
    for dt in pd.date_range(start_dt, end_dt, freq="D")
]

src = pd.concat(load_observation_files(daily_paths), ignore_index=True)

# Normalize free-text columns: keep only the IP portion of the indicator field
# and strip stray whitespace from the OpDiv code.
if "indicator" in src.columns:
    src["indicator"] = src["indicator"].astype(str).str.split(" ", expand=True)[0].str.strip()
if "OpDiv" in src.columns:
    src["OpDiv"] = src["OpDiv"].astype(str).str.strip()

display(src)


,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,101.89.174.236,15920309593055310684,2025-07-09,CMS,101.89.174.236_CMS,2106,2025-07-09
1,101.89.174.236,13983566077459384977,2025-07-09,FDA,101.89.174.236_FDA,327,2025-07-09
2,103.120.176.224,74198686107399967946,2025-07-09,VA,103.120.176.224_VA,3,2025-07-09
3,103.125.189.6,13983566077459384977,2025-07-09,FDA,103.125.189.6_FDA,3339,2025-07-09
4,103.149.86.208,15920309593055310684,2025-07-09,CMS,103.149.86.208_CMS,570,2025-07-09
...,...,...,...,...,...,...,...
441523,159.65.6.6,74198686107399967946,2025-10-17,VA,159.65.6.6_VA,1,2025-10-17
441524,193.32.162.21,13520457953477229971,2025-10-17,HHS,193.32.162.21_HHS,1,2025-10-17
441525,104.152.52.216,74198686107399967946,2025-10-17,VA,104.152.52.216_VA,1,2025-10-17
441526,104.152.52.239,74198686107399967946,2025-10-17,VA,104.152.52.239_VA,1,2025-10-17


In [ ]:
# Keep one tidy observation table: indicator / source / date / OpDiv / count.
src = (
    src.drop(columns=["curr_date", "indicator_key"])
       .rename(columns={"obs_date": "date"})
       .assign(date=lambda d: pd.to_datetime(d["date"]))
       .reset_index(drop=True)
)
src


,indicator,API_UserName,date,OpDiv,observations
0,102.90.61.13,64853912235916233429,2025-03-16,OS,3117
1,102.91.94.193,64853912235916233429,2025-03-16,OS,2195
2,103.225.136.166,64853912235916233429,2025-03-16,OS,4
3,104.18.68.40,15920309593055310684,2025-03-16,CMS,5
4,104.18.69.40,15920309593055310684,2025-03-16,CMS,5
...,...,...,...,...,...
81418,104.152.52.148,74198686107399967946,2025-06-24,VA,1
81419,159.13.45.83,20790633968691748718,2025-06-24,DHA,1
81420,45.138.16.240,64853912235916233429,2025-06-24,OS,1
81421,196.251.70.216,64853912235916233429,2025-06-24,OS,1


## 3 · Spot checks

Quick lookup helper used throughout development to trace a single indicator's history.


In [ ]:
opdiv_groups = dict(tuple(src.groupby("OpDiv")))


def get_by_indicator(indicator_value):
    """Return every observation of `indicator_value` across all OpDivs."""
    return src[src["indicator"] == indicator_value]


get_by_indicator("192.124.249.112")


,indicator,API_UserName,date,OpDiv,observations
11221,192.124.249.112,50189120947314147395,2025-04-15,NIH,4
11910,192.124.249.112,80363974983666420473,2025-04-16,IHS,13
11911,192.124.249.112,50189120947314147395,2025-04-16,NIH,151
11912,192.124.249.112,74198686107399967946,2025-04-16,VA,14
12649,192.124.249.112,00818860012482918321,2025-04-17,CDC,7
...,...,...,...,...,...
69013,192.124.249.112,74198686107399967946,2025-06-18,VA,29
72802,192.124.249.112,50189120947314147395,2025-06-20,NIH,7
76778,192.124.249.112,50189120947314147395,2025-06-22,NIH,3
78620,192.124.249.112,00818860012482918321,2025-06-23,CDC,44


## 4 · Dense panel per OpDiv

The raw extracts only contain rows for days an indicator **was** observed. The models
need the silent days too, so for each OpDiv we build the full
`source x date x indicator` grid, zero-fill the missing observations, and derive the
binary target `seen` plus calendar features.


In [ ]:
def build_dense_panel(group_df):
    """Expand one OpDiv's observations into a complete daily grid.

    Every (source, date, indicator) combination from the OpDiv's first observed
    date through today gets a row; days without an observation become zeros.
    """
    group_df = group_df.assign(date=pd.to_datetime(group_df["date"]))

    grid = pd.MultiIndex.from_product(
        [
            group_df["API_UserName"].unique(),
            pd.date_range(group_df["date"].min(), pd.Timestamp.now().normalize(), freq="D"),
            group_df["indicator"].unique(),
        ],
        names=["API_UserName", "date", "indicator"],
    ).to_frame(index=False)
    grid["OpDiv"] = group_df["OpDiv"].iloc[0]

    panel = grid.merge(group_df, how="left", on=["API_UserName", "date", "indicator", "OpDiv"])
    panel["observations"] = panel["observations"].fillna(0).astype(int)

    # Calendar features + binary target
    panel["dayofweek"] = panel["date"].dt.dayofweek
    panel["is_weekend"] = panel["dayofweek"].isin([5, 6])
    panel["day"] = panel["date"].dt.day
    panel["month"] = panel["date"].dt.month
    panel["seen"] = (panel["observations"] > 0).astype(int)
    return panel


opdiv_panels = {opdiv: build_dense_panel(df) for opdiv, df in opdiv_groups.items()}

display(opdiv_panels["DHA"])


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
0,20790633968691748718,2025-07-09,103.203.59.0,DHA,1,2,False,9,7,1
1,20790633968691748718,2025-07-09,103.61.44.100,DHA,1,2,False,9,7,1
2,20790633968691748718,2025-07-09,104.131.6.219,DHA,1024,2,False,9,7,1
3,20790633968691748718,2025-07-09,104.160.6.2,DHA,2,2,False,9,7,1
4,20790633968691748718,2025-07-09,104.234.115.162,DHA,3012,2,False,9,7,1
...,...,...,...,...,...,...,...,...,...,...
156444,20790633968691748718,2025-10-17,77.111.247.82,DHA,0,4,False,17,10,0
156445,20790633968691748718,2025-10-17,nintenduo.com,DHA,0,4,False,17,10,0
156446,20790633968691748718,2025-10-17,103.167.91.247,DHA,6,4,False,17,10,1
156447,20790633968691748718,2025-10-17,164.52.24.187,DHA,3,4,False,17,10,1


In [ ]:
# Sanity check: the panel now carries explicit zero-observation days for this indicator.
opdiv_panels["VA"][opdiv_panels["VA"]["indicator"] == "192.124.249.112"]


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
144,74198686107399967946,2025-03-21,192.124.249.112,VA,0,4,False,21,3,0
1026,74198686107399967946,2025-03-22,192.124.249.112,VA,0,5,True,22,3,0
1908,74198686107399967946,2025-03-23,192.124.249.112,VA,0,6,True,23,3,0
2790,74198686107399967946,2025-03-24,192.124.249.112,VA,0,0,False,24,3,0
3672,74198686107399967946,2025-03-25,192.124.249.112,VA,0,1,False,25,3,0
...,...,...,...,...,...,...,...,...,...,...
80406,74198686107399967946,2025-06-20,192.124.249.112,VA,0,4,False,20,6,0
81288,74198686107399967946,2025-06-21,192.124.249.112,VA,0,5,True,21,6,0
82170,74198686107399967946,2025-06-22,192.124.249.112,VA,0,6,True,22,6,0
83052,74198686107399967946,2025-06-23,192.124.249.112,VA,0,0,False,23,6,0


## 5 · Feature engineering

Per indicator, from its daily `seen` series (oldest → newest):

- **`last_seen`** — days since the most recent observation.
- **`freq_w`** — observation-day counts over trailing windows *w* ∈ {1, 7, 14, 30, 45}.
- **`avg_gap`** — mean gap (days) between consecutive observations.
- **`burstiness`** — (σ−μ)/(σ+μ) of the gaps: −1 for perfectly periodic,
  → +1 for highly bursty behavior.
- **`label_w`** — whether the indicator was seen at all in the trailing window,
  used as the training target per horizon.


In [ ]:
FEATURE_COLS = ["last_seen", "freq_1", "freq_7", "freq_14", "freq_30", "freq_45",
                "avg_gap", "burstiness"]


def extract_time_series_features(group):
    """Collapse one indicator's daily `seen` series into a feature row."""
    series = group["seen"].to_numpy()
    seen_idx = np.flatnonzero(series)

    if seen_idx.size == 0:
        feats = {"last_seen": len(series), "avg_gap": len(series), "burstiness": 0}
        feats.update({f"freq_{w}": 0 for w in HORIZONS})
        feats.update({f"label_{w}": 0 for w in LABEL_HORIZONS})
        return pd.Series(feats)

    gaps = np.diff(seen_idx)
    avg_gap = gaps.mean() if gaps.size > 0 else len(series)

    feats = {
        "last_seen": len(series) - 1 - seen_idx[-1],
        "avg_gap": avg_gap,
        "burstiness": (gaps.std() - avg_gap) / (gaps.std() + avg_gap) if gaps.size > 1 else 0,
    }
    feats.update({f"freq_{w}": int(series[-w:].sum()) for w in HORIZONS})
    feats.update({f"label_{w}": int(series[-w:].any()) for w in LABEL_HORIZONS})
    return pd.Series(feats)


def build_features(panel):
    """One feature row per indicator in the OpDiv panel."""
    return panel.groupby("indicator").apply(extract_time_series_features).reset_index()


## 6 · Probability models

Each model is fit per OpDiv on the current snapshot to produce a relative
likelihood ranking for analysts each morning. The 1-day logistic/GBT scores
train against the 7-day label, while the ensemble's Weibull and exponential
terms model that horizon directly.


In [ ]:
def make_logistic():
    """Logistic regression with feature scaling (scale-sensitive model)."""
    return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())])


def train_predict_proba(make_model, X, y):
    """Fit a fresh classifier and return P(y=1); NaN if `y` is single-class."""
    if y.nunique() < 2:
        return np.full(len(y), np.nan)
    model = make_model()
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]


def fit_weibull_aft(X, durations, events):
    """Weibull accelerated-failure-time model over inter-observation gaps."""
    aft_df = X.assign(duration=durations, event=events)
    aft = WeibullAFTFitter()
    aft.fit(aft_df, duration_col="duration", event_col="event")
    return aft


def get_model_outputs(features_df, panel):
    """Score every indicator with all four models across all horizons."""
    scores = features_df.copy()
    X = scores[FEATURE_COLS]

    # Logistic regression + gradient-boosted trees, one target per horizon
    for h in LABEL_HORIZONS:
        y = scores[f"label_{h}"]
        scores[f"logistic_{h}"] = train_predict_proba(make_logistic, X, y)
        scores[f"gbt_{h}"] = train_predict_proba(GradientBoostingClassifier, X, y)

    # 1-day horizon: 7-day labels as proxy (see section note)
    scores["logistic_1"] = train_predict_proba(make_logistic, X, scores["label_7"])
    scores["gbt_1"] = train_predict_proba(GradientBoostingClassifier, X, scores["label_7"])

    # Exponential model: memoryless Poisson process on the trailing 30-day rate
    rate = (scores["freq_30"] / 30).clip(lower=1e-6)
    for h in HORIZONS:
        scores[f"exp_{h}"] = 1 - np.exp(-rate * h)

    # Weibull AFT: survival curve over the average inter-observation gap
    aft = fit_weibull_aft(X, scores["avg_gap"], scores["label_7"])
    surv = aft.predict_survival_function(
        X.assign(duration=scores["avg_gap"], event=scores["label_7"]), times=HORIZONS
    )
    for h in HORIZONS:
        scores[f"weibull_{h}"] = 1 - surv.loc[h].values

    # Attach whether the indicator was actually seen on the latest date
    latest = panel["date"].max()
    seen_today = (
        panel.loc[panel["date"] == latest, ["indicator", "seen"]]
             .rename(columns={"seen": "seen_today"})
    )
    return scores.merge(seen_today, on="indicator", how="left")


## 7 · Rule baseline, ensemble, and confidence tiers

A recency rule (`last_seen < h`) provides deterministic labels; a logistic model fit
against those labels smooths them into probabilities, which are blended with the GBT,
Weibull, and exponential scores using fixed weights. The blend is then bucketed into
analyst-facing confidence tiers, gated on recent frequency so that a high probability
with no recent activity is not over-sold.


In [ ]:
RULE_FEATURES = ["last_seen", "freq_1", "freq_7", "freq_30", "avg_gap", "burstiness"]


def add_rule_and_ensemble(output):
    """Add rule labels, rule-smoothed probabilities, and the weighted ensemble."""
    X = output[RULE_FEATURES]

    for h in HORIZONS:
        output[f"rule_{h}d"] = (output["last_seen"] <= h - 1).astype(int)
        output[f"prob_{h}d"] = train_predict_proba(make_logistic, X, output[f"rule_{h}d"])
        output[f"ensemble_{h}d"] = (
            ENSEMBLE_WEIGHTS["logistic"] * output[f"prob_{h}d"].astype(float)
            + ENSEMBLE_WEIGHTS["gbt"] * output[f"gbt_{h}"]
            + ENSEMBLE_WEIGHTS["weibull"] * output[f"weibull_{h}"]
            + ENSEMBLE_WEIGHTS["exponential"] * output[f"exp_{h}"]
        )
    return output


def add_confidence_and_format(output):
    """Bucket ensemble scores into confidence tiers and format probabilities."""
    for h in HORIZONS:
        prob = output[f"ensemble_{h}d"].astype(float)
        freq = output[f"freq_{h}"]
        label = f"{h}-Day"
        output[f"confidence_{h}d"] = np.select(
            [(prob >= 0.6) & (freq >= 2), (prob >= 0.07) & (freq >= 1)],
            [f"{label}: Highly likely", f"{label}: Possibly active"],
            default=f"{label}: Low confidence",
        )

    prob_cols = [f"prob_{h}d" for h in HORIZONS] + [f"ensemble_{h}d" for h in HORIZONS]
    for col in prob_cols:
        output[col] = np.clip(output[col].astype(float) * 100, 0, 100).round(2).astype(str) + "%"
    return output


def build_production_output(output):
    """Select and rename the analyst-facing columns."""
    production = output[
        ["indicator", "seen_today", "freq_1", "freq_7", "freq_30"]
        + [c for h in HORIZONS for c in (f"ensemble_{h}d", f"confidence_{h}d")]
    ].copy()
    return production.rename(columns={
        "indicator": "Indicator",
        "seen_today": "Observed Today",
        "freq_1": "Frequency (1d)",
        "freq_7": "Frequency (7d)",
        "freq_30": "Frequency (30d)",
        **{f"ensemble_{h}d": f"Probability: {h}-Day" for h in HORIZONS},
        **{f"confidence_{h}d": f"Confidence: {h}-Day" for h in HORIZONS},
    })


## 8 · Run the pipeline per OpDiv

In [ ]:
def run_pipeline(panels):
    """Features -> model scores -> ensemble -> production table, per OpDiv."""
    forecasts = {}
    for opdiv, panel in panels.items():
        features_df = build_features(panel)
        output = get_model_outputs(features_df, panel)
        output = add_rule_and_ensemble(output)
        output = add_confidence_and_format(output)
        forecasts[opdiv] = build_production_output(output)
    return forecasts


forecasts = run_pipeline(opdiv_panels)

sample_opdiv = list(forecasts)[-1]
display(forecasts[sample_opdiv].head(5))
display(opdiv_panels[sample_opdiv].head(5))


,Indicator,Observed Today,Frequency (1d),Frequency (7d),Frequency (30d),Probability: 1-Day,Confidence: 1-Day,Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,ensemble_45d,Confidence: 45-Day
0,1-you.njalla.no,0,0.0,0.0,0.0,0.02%,1-Day: Low confidence,0.14%,7-Day: Low confidence,0.17%,14-Day: Low confidence,13.71%,30-Day: Low confidence,29.64%,45-Day: Possibly active
1,1.4.195.14,0,0.0,0.0,7.0,4.2%,1-Day: Low confidence,21.06%,7-Day: Low confidence,90.13%,14-Day: Highly likely,99.97%,30-Day: Highly likely,76.04%,45-Day: Highly likely
2,100.27.42.247,0,0.0,0.0,0.0,0.02%,1-Day: Low confidence,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.17%,30-Day: Low confidence,8.7%,45-Day: Low confidence
3,102.0.5.152,0,0.0,1.0,2.0,26.35%,1-Day: Low confidence,53.56%,7-Day: Possibly active,77.44%,14-Day: Possibly active,97.26%,30-Day: Highly likely,74.47%,45-Day: Highly likely
4,102.164.252.150,0,0.0,0.0,0.0,0.0%,1-Day: Low confidence,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.0%,30-Day: Low confidence,0.1%,45-Day: Low confidence


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
0,74198686107399967946,2025-07-09,103.120.176.224,VA,3,2,False,9,7,1
1,74198686107399967946,2025-07-09,103.149.86.208,VA,4,2,False,9,7,1
2,74198686107399967946,2025-07-09,103.61.44.100,VA,1,2,False,9,7,1
3,74198686107399967946,2025-07-09,104.128.161.233,VA,3,2,False,9,7,1
4,74198686107399967946,2025-07-09,104.152.52.105,VA,2,2,False,9,7,1


## Appendix · Feedback loop and per-OpDiv retraining

Forecasts are logged and later graded by analysts (`Seen` / `Not Seen`). This section
folds those graded outcomes back into a per-OpDiv master training set and retrains the
7-day gradient-boosted model, so the system improves as ground truth accumulates.
*Not executed in this snapshot — it expects the graded `forecast_log.xlsx` files
produced by the batch deployment.*


In [ ]:
def load_graded_forecast_logs(logs_base):
    """Collect graded (non-pending) forecast logs from each OpDiv subfolder."""
    logs = {}
    for sub in logs_base.iterdir():
        log_file = sub / "forecast_log.xlsx"
        if log_file.exists():
            df = pd.read_excel(log_file)
            logs[sub.name] = df[df["Outcome"].str.lower() != "pending"]
    if not logs:
        raise FileNotFoundError(f"No forecast_log.xlsx found under {logs_base}")
    return logs


def update_master_and_retrain(opdiv, log_df):
    """Append graded outcomes to the OpDiv's master set and retrain its 7-day GBT."""
    feats = (
        log_df.groupby(["Indicator", "ForecastDate"])
              .apply(extract_time_series_features)
              .reset_index(drop=True)
    )
    feats["y_true_7d"] = (log_df["Outcome"] == "Seen").astype(int)

    master_path = Path(f"train_master_{opdiv}.csv")
    master = pd.concat([pd.read_csv(master_path), feats], ignore_index=True) \
        if master_path.exists() else feats.copy()
    master.drop_duplicates(subset=["Indicator", "ForecastDate"], keep="last", inplace=True)
    master.to_csv(master_path, index=False)
    print(f"Updated master for {opdiv}: {len(master)} rows saved to {master_path}")

    feature_cols = [c for c in master.columns if c not in ("Indicator", "ForecastDate", "y_true_7d")]
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=3)
    clf.fit(master[feature_cols], master["y_true_7d"])

    model_file = f"gbc_7d_{opdiv}.pkl"
    with open(model_file, "wb") as f:
        pickle.dump(clf, f)
    print(f"Retrained and saved model for {opdiv} -> {model_file}")


opdiv_logs = load_graded_forecast_logs(Path("Logs"))
print(f"Found logs for OPDIVs: {list(opdiv_logs.keys())}")

for opdiv, log_df in opdiv_logs.items():
    print(f"\nProcessing {opdiv}, {len(log_df)} records...")
    update_master_and_retrain(opdiv, log_df)
